In [ ]:
"""
Project 01: Credit Risk Prediction
Step 04: Model Training
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import json
import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)
import lightgbm as lgb
import xgboost as xgb

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in globals() else (os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 04: Model Training")
print("=" * 60)
start_time = time.time()

In [ ]:
# ── Load Preprocessed Data ──
print("\nLoading preprocessed data...")
df = pd.read_csv(os.path.join(PROCESSED, "preprocessed.csv"))
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"
X = df.drop(columns=[TARGET, ID_COL], errors="ignore")
y = df[TARGET]

In [ ]:
print(f"Target distribution:\n{y.value_counts()}")
print(f"Default rate: {y.mean():.2%}")

In [ ]:
# ── Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

In [ ]:
# ── Scale ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
feature_names = X.columns.tolist()

In [ ]:
def evaluate(name, model, X_te, y_te):
    """Evaluate model and return metrics dict."""
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    metrics = {
        "Model": name,
        "auroc": round(roc_auc_score(y_te, y_proba), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_te, y_pred), 4),
        "f1": round(f1_score(y_te, y_pred), 4),
        "precision": round(precision_score(y_te, y_pred), 4),
        "recall": round(recall_score(y_te, y_pred), 4),
    }
    print(f"\n  {name}:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"    {k}: {v}")
    return metrics, y_proba

In [ ]:
# ── Model 1: Logistic Regression ──
print("\n[1/3] Training Logistic Regression...")
t0 = time.time()
lr = LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=42, solver="lbfgs"
)
lr.fit(X_train_scaled, y_train)
lr_time = round(time.time() - t0, 2)
metrics_lr, proba_lr = evaluate("Logistic Regression", lr, X_test_scaled, y_test)
metrics_lr["train_time_sec"] = lr_time

In [ ]:
with open(os.path.join(MODELS, "logistic_regression.pkl"), "wb") as f:
    pickle.dump({"model": lr, "scaler": scaler, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {lr_time}s")

In [ ]:
# ── Model 2: LightGBM ──
print("\n[2/3] Training LightGBM...")
t0 = time.time()
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1,
)
lgbm.fit(X_train, y_train)
lgb_time = round(time.time() - t0, 2)
metrics_lgb, proba_lgb = evaluate("LightGBM", lgbm, X_test, y_test)
metrics_lgb["train_time_sec"] = lgb_time

In [ ]:
with open(os.path.join(MODELS, "lightgbm.pkl"), "wb") as f:
    pickle.dump({"model": lgbm, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {lgb_time}s")

In [ ]:
# ── Model 3: XGBoost ──
print("\n[3/3] Training XGBoost...")
t0 = time.time()
xgbm = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_pos,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss",
    verbosity=0,
)
xgbm.fit(X_train, y_train)
xgb_time = round(time.time() - t0, 2)
metrics_xgb, proba_xgb = evaluate("XGBoost", xgbm, X_test, y_test)
metrics_xgb["train_time_sec"] = xgb_time

In [ ]:
with open(os.path.join(MODELS, "xgboost.pkl"), "wb") as f:
    pickle.dump({"model": xgbm, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {xgb_time}s")

In [ ]:
# ── Save Results ──
results = {
    "Logistic Regression": metrics_lr,
    "LightGBM": metrics_lgb,
    "XGBoost": metrics_xgb,
}
with open(os.path.join(MODELS, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
# ── Charts: Model Comparison ──
print("\nGenerating model comparison chart...")
df_res = pd.DataFrame([metrics_lr, metrics_lgb, metrics_xgb])
df_res = df_res.sort_values("auroc", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = ["#6C63FF", "#FF6584", "#43E97B"]

In [ ]:
for ax, metric, title in zip(
    axes,
    ["auroc", "f1", "precision", "recall"],
    ["AUROC", "F1 Score", "Precision", "Recall"],
):
    bars = ax.bar(df_res["Model"], df_res[metric], color=colors, alpha=0.85)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, df_res[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.3f}",
            ha="center",
            fontsize=10,
            fontweight="bold",
        )
    ax.tick_params(axis="x", rotation=30)

In [ ]:
plt.suptitle("Model Comparison — Credit Risk", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: model_comparison.png")

In [ ]:
# ── Feature Importance (LightGBM) ──
print("Generating feature importance chart...")
imp = (
    pd.DataFrame({"feature": feature_names, "importance": lgbm.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(20)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp["feature"], imp["importance"], color="#6C63FF", alpha=0.85)
ax.set_xlabel("Importance")
ax.set_title("Top 20 Features — LightGBM", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "feature_importance.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print("  Saved: feature_importance.png")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 04 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")